In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import torch
from omegaconf import OmegaConf

from src.builder import (
    build_critic,
    build_generator,
    build_image_loader,
    build_optimizer,
    build_trainer,
    validate_config,
)

In [2]:
cfg = OmegaConf.load(REPO_ROOT / "configs" / "default.yaml")
cfg.trainer.steps = 10
cfg.trainer.save_freq = 10
if not torch.cuda.is_available():
    cfg.device = "cpu"
validate_config(cfg)

device = torch.device(cfg.device)
netG = build_generator(cfg).to(device)
netCs = [build_critic(cfg).to(device) for _ in range(3)]
optG = build_optimizer(cfg, netG.parameters())
optCs = [build_optimizer(cfg, c.parameters()) for c in netCs]
image_loader = build_image_loader(cfg)

trainer = build_trainer(cfg, netG, netCs, optG, optCs, image_loader)
print("device   :", device)
print("save_dir :", trainer.save_dir)

device   : cuda
save_dir : run/20260421_204004


In [3]:
for step in range(cfg.trainer.steps):
    losses = trainer.step(step)
    print(f"step {step:02d} | " + "  ".join(f"{k}={v:+.4f}" for k, v in losses.items()))
trainer.save()
trainer.writer.flush()
trainer.writer.close()

step 00 | critic_fake_score=-0.0044  critic_real_score=-0.0083  wass_dist=-0.0039  gp=+9.5767  loss=+9.5806
step 01 | critic_fake_score=-0.0030  critic_real_score=-0.0056  wass_dist=-0.0026  gp=+9.5566  loss=+9.5593
step 02 | critic_fake_score=+0.0037  critic_real_score=+0.0059  wass_dist=+0.0022  gp=+9.5667  loss=+9.5644
step 03 | critic_fake_score=-0.0014  critic_real_score=+0.0001  wass_dist=+0.0015  gp=+9.5539  loss=+9.5524
step 04 | critic_fake_score=+0.0000  critic_real_score=+0.0021  wass_dist=+0.0021  gp=+9.5333  loss=+9.5312
step 05 | critic_fake_score=+0.0079  critic_real_score=+0.0176  wass_dist=+0.0097  gp=+9.5419  loss=+9.5321  generator_loss=-0.0055  adv_loss=-0.0055  recon_loss=+0.0000
step 06 | critic_fake_score=+0.0037  critic_real_score=+0.0097  wass_dist=+0.0060  gp=+9.5308  loss=+9.5248
step 07 | critic_fake_score=+0.0059  critic_real_score=+0.0104  wass_dist=+0.0045  gp=+9.4994  loss=+9.4950
step 08 | critic_fake_score=+0.0195  critic_real_score=+0.0302  wass_dist=